In [28]:
import numpy as np
import tensorflow as tf
import pandas as pd

In [29]:
# Load data and train the model
df = pd.read_csv('df_with_stability.csv')

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144976 entries, 0 to 144975
Data columns (total 7 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   age                    144976 non-null  int64  
 1   gender                 144976 non-null  int64  
 2   oxygen_rate            144976 non-null  float64
 3   heart_rate             144976 non-null  float64
 4   heart_rate_stability   144976 non-null  int64  
 5   oxygen_rate_stability  144976 non-null  int64  
 6   stability              144976 non-null  int64  
dtypes: float64(2), int64(5)
memory usage: 7.7 MB


In [31]:
# Define features and target
features = df[['age', 'heart_rate', 'oxygen_rate']]
target = df['stability']


In [32]:
# Split data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.3)


In [33]:
# Create and compile the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(3,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\core\dense.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.6932 - loss: 0.7469 - val_accuracy: 0.7675 - val_loss: 0.5100
Epoch 2/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7670 - loss: 0.5189 - val_accuracy: 0.7894 - val_loss: 0.4725
Epoch 3/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8132 - loss: 0.4524 - val_accuracy: 0.8261 - val_loss: 0.4902
Epoch 4/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8502 - loss: 0.3904 - val_accuracy: 0.8874 - val_loss: 0.3288
Epoch 5/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8770 - loss: 0.3329 - val_accuracy: 0.8828 - val_loss: 0.2836
Epoch 6/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8967 - loss: 0.2897 - val_accuracy: 0.9113 - val_loss: 0.2618
Epoch 7/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9044 - loss: 0.2672 - val_accuracy: 0.9270 - val_loss: 0.2440
Epoch 8/10
3172/3172 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9095 - loss: 0.2467 - 

In [35]:
# Evaluate the model on the test set
loss, mae = model.evaluate(X_test, y_test)
print(f"Mean Absolute Error: {mae}")
print(f"Loss: {loss}")


1360/1360 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9172 - loss: 0.2170
Mean Absolute Error: 0.9139401912689209
Loss: 0.22558113932609558


In [36]:
# Make predictions on new data
y_pred = model.predict(X_test)


1360/1360 ━━━━━━━━━━━━━━━━━━━━ 1s 725us/step


In [41]:
# Get user input
age = 30
heart_rate = 98.0
oxygen_rate = 96.0


In [ ]:

# Convert input to NumPy array
new_data = np.array([[age, heart_rate, oxygen_rate]])

# Make prediction
predicted_class = model.predict(new_data)

# Get the class with the highest probability
predicted_class = np.argmax(predicted_class, axis=-1)


def predict_status(predicted_class):
  """Maps the predicted class to the corresponding status."""
  if predicted_class == 0:
    return 'Stable'
  elif predicted_class == 1:
    return 'Instable'
  else:
    return 'Grave'

  
# Get the predicted status
predicted_status = predict_status(predicted_class[0])  



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step


In [39]:
# Print the status
print("Predicted Status:", predicted_status)

Predicted Status: Stable


In [40]:
# Convert and save the model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('stability1.tflite', 'wb') as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\pc\AppData\Local\Temp\tmpwkou3hc8\assets


INFO:tensorflow:Assets written to: C:\Users\pc\AppData\Local\Temp\tmpwkou3hc8\assets


Saved artifact at 'C:\Users\pc\AppData\Local\Temp\tmpwkou3hc8'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2467039156304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2467039165712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2467057942992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2467057947792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2467057945104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2467057943568: TensorSpec(shape=(), dtype=tf.resource, name=None)
